# Reading Code Easily: Python Patterns in Compass

This notebook is a guided tour of the coding and design patterns that make a
codebase pleasant to read. Every example is taken from **compass**, the library
in this repository for evaluating subjective LLM behavior, so the patterns are
not abstract advice. You can open the referenced source file next to each
section and see them in context.

The thesis is simple: readable code is not code with more comments. It is code
where the *shape* of the program tells you what it does, where names carry
meaning, where the type system and the structure rule out whole classes of
confusion before you ever read a comment. Comments then explain the one thing
the code cannot: **why**.

## How to use this notebook

Run the setup cells first, in order. The first one bootstraps the environment so
the notebook works both in a local checkout and on **Google Colab**. After that,
every code cell stands alone and runs offline. Where a pattern needs a live
model we substitute a small fake client so you can watch the real control flow
without an API key.

## What we cover

1. Frozen dataclasses as value objects
2. Content-addressed identity through computed properties
3. Docstrings written for the caller, then the maintainer
4. Computed properties versus stored fields
5. Contracts: abstract base classes and Protocols
6. EAFP, the Pythonic way to handle the missing case
7. Defensive parsing with explicit, ordered fallbacks
8. Validation and normalization in `__post_init__`
9. Deep immutability with `MappingProxyType` and `object.__setattr__`
10. Intention-revealing names
11. Small functions with one job
12. Comments that explain *why*, and invariants stated once
13. Dependency injection
14. Lazy imports: `TYPE_CHECKING` and deferred imports
15. Atomic file writes
16. The registry pattern with registration-time validation
17. A capstone that ties the judge, client, and cache together
18. A readability checklist

Let's begin.

## Setup (local and Google Colab)

This cell makes the notebook portable. It works as-is in a local checkout where
`compass` is already importable, and it bootstraps a **Google Colab** session
where you have cloned the repository.

On Colab the usual flow is:

```python
!git clone <your-compass-repo-url>.git
```

then run this cell. It searches the common locations for a `compass` checkout (a
directory containing `setup.py`), installs it in editable mode with
`pip install -e .`, and verifies the import. If your clone lives somewhere
unusual, set `REPO_DIR` to its path before running.

In [ ]:
import importlib, os, subprocess, sys
from pathlib import Path

# If you cloned the repo to a non-standard location on Colab, set it here.
REPO_DIR = os.environ.get("COMPASS_REPO_DIR")  # e.g. "/content/compass"

IN_COLAB = "google.colab" in sys.modules


def _find_checkout() -> Path | None:
    "Locate a compass source checkout by its setup.py."
    candidates = []
    if REPO_DIR:
        candidates.append(Path(REPO_DIR))
    candidates += [
        Path.cwd(),
        Path.cwd() / "compass",
        Path("/content/compass"),      # default Colab clone target
        Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd(),
    ]
    for c in candidates:
        if (c / "setup.py").exists() and (c / "compass" / "__init__.py").exists():
            return c
    return None


def _ensure_compass() -> None:
    try:
        import compass  # noqa: F401
        return
    except ImportError:
        pass

    checkout = _find_checkout()
    if checkout is None:
        raise ImportError(
            "compass is not importable and no checkout was found.\n"
            "On Colab, clone it first, e.g.:\n"
            "    !git clone <your-compass-repo-url>.git\n"
            "then set COMPASS_REPO_DIR to the clone path and re-run this cell."
        )
    print(f"Installing compass from {checkout} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(checkout)])
    importlib.invalidate_caches()


_ensure_compass()

import compass
print("environment :", "Google Colab" if IN_COLAB else "local")
print("compass     :", compass.__version__)
print("python      :", sys.version.split()[0])

## Imports we will study

We import the pieces of compass this notebook dissects. None of these touch the
network or require credentials.

In [ ]:
import sys
import compass

print("compass", compass.__version__)
print("python ", sys.version.split()[0])

# The public surface of the library, re-exported from compass/__init__.py.
# A curated __all__ like this one is itself a readability pattern: it tells a
# reader exactly which names are the supported API and which are internal.
print("\nPublic API names:")
print(", ".join(n for n in compass.__all__ if not n.startswith("__")))

## 1. Frozen dataclasses as value objects

`compass/rubrics/base.py` defines a `Rubric` as a *frozen* dataclass. A rubric
is a value: two rubrics with the same fields are the same rubric, and a rubric
should never change after creation. Freezing the dataclass makes that intent
enforceable rather than merely documented.

```python
@dataclass(frozen=True)
class Rubric:
    name: str          # e.g., "sycophancy"
    category: str      # e.g., "constitutional", "alignment", "safety"
    version: str       # e.g., "1.0", "1.1" (semantic)
    created_at: str    # ISO timestamp
    text: str          # The actual rubric prompt given to the judge
    hit_threshold: float = 0.5
    max_tokens: int = 180
```

Why this reads well:

- **`@dataclass` removes boilerplate.** No hand-written `__init__`, `__eq__`, or
  `__repr__` to scan past. The field list *is* the definition.
- **`frozen=True` states a guarantee.** A reader knows immediately that a
  `Rubric` cannot be mutated, so they never have to hunt for code that might
  change one out from under them. The cache and reproducibility logic depend on
  this; freezing is what makes the dependency safe.
- **Inline field comments give a concrete example for each field.** `name` is
  not just a `str`; it is `"sycophancy"`. One example is worth a paragraph.

Run the cell to see immutability enforced at runtime.

In [ ]:
from dataclasses import FrozenInstanceError
from compass.rubrics.base import Rubric

r = Rubric(
    name="sycophancy",
    category="constitutional",
    version="1.0",
    created_at="2026-05-28",
    text="Score 1.0 when the response flatters instead of correcting...",
)

print("repr:", repr(r))          # custom __repr__ keeps logs readable
print("equal to a twin:", r == Rubric(**{**r.__dict__}))  # value equality, free

# Freezing turns an accidental mutation into a loud, immediate error.
try:
    r.hit_threshold = 0.9
except FrozenInstanceError as e:
    print("mutation blocked:", e)

## 2. Content-addressed identity through computed properties

A `Rubric` carries no stored ID. Its identity is *derived* from its content:

```python
@property
def hash(self) -> str:
    \"\"\"Immutable hash of this rubric. Changes if content changes.\"\"\"
    content = f"{self.name}_{self.version}_{self.text}_{self.hit_threshold}"
    return hashlib.sha256(content.encode()).hexdigest()[:16]

@property
def full_id(self) -> str:
    \"\"\"Human-readable ID combining name, version, and hash.\"\"\"
    return f"{self.name}_v{self.version}_{self.hash}"
```

This is the **content-addressed** pattern. Because the hash is a pure function
of the fields and the dataclass is frozen, the hash is stable for the life of
the object and identical across processes and machines. That is exactly what a
cache key and a reproducibility identifier need.

Why it reads well: there is no separate "assign an ID" step to find and reason
about, and no way for the ID to drift out of sync with the content. The
invariant "same content implies same id" is not a comment you have to trust; it
is structurally true because `hash` reads the fields every time and the fields
cannot change.

In [ ]:
r1 = Rubric("sycophancy", "constitutional", "1.0", "2026-05-28", "Score 1.0 when...")
r2 = Rubric("sycophancy", "constitutional", "1.0", "2026-05-28", "Score 1.0 when...")
r3 = Rubric("sycophancy", "constitutional", "1.1", "2026-05-28", "Score 1.0 when...")

print("r1.hash    ", r1.hash)
print("r2.hash    ", r2.hash, "(same content -> same hash)")
print("r3.hash    ", r3.hash, "(version bumped -> different hash)")
print("r1.full_id ", r1.full_id)
assert r1.hash == r2.hash and r1.hash != r3.hash

## 3. Docstrings written for the caller, then the maintainer

Look at `EvaluationResult` in `compass/judges/base.py`. Its docstring leads with
an `Attributes:` block that explains each field in plain language, then closes
with a `Note:` aimed at someone trying to reproduce a result. The team
convention (see the project `CLAUDE.md`) is explicit about this ordering: *lead
with usage details for callers, then non-trivial notes for maintainers.*

The payoff is that `help(EvaluationResult)` is a real reference. A reader does
not have to open the source to learn that `score` is clamped to `[0, 1]`, that
`hit` is `score >= threshold`, or that the cache key is
`(config_hash, text_hash, prompt_version)`.

Run the cell and read the rendered docstring as a user would.

In [ ]:
from compass.judges.base import EvaluationResult
print(EvaluationResult.__doc__)

## 4. Computed properties versus stored fields

When a value can be derived, compass derives it instead of storing it. Storing
it would invite the two copies to disagree. `JudgeConfig.config_hash` and the
client cost accessors are computed on read:

```python
@property
def config_hash(self) -> str:
    content = f"{self.rubric.hash}_{self.judge_model}_{self.max_tokens}_{self.temperature}_{self.seed}"
    return hashlib.sha256(content.encode()).hexdigest()[:12]
```

```python
@property
def total_cost_usd(self) -> float:
    return (
        self._input_tokens * self._pricing.input_cost_per_million / 1_000_000
        + self._output_tokens * self._pricing.output_cost_per_million / 1_000_000
    )
```

The rule of thumb: **store inputs, compute outputs.** A property reads like an
attribute at the call site (`config.config_hash`, `client.total_cost_usd`) so
it stays clean, but it can never go stale because there is nothing to keep in
sync. When the underlying `_input_tokens` changes, the cost reflects it on the
next read.

In [ ]:
from compass.judges.base import JudgeConfig

cfg = JudgeConfig(rubric=r1, judge_model="claude-haiku-4-5")
print("config_hash:", cfg.config_hash)
print("repr       :", repr(cfg))

# Stability check: a second config with identical inputs hashes identically.
cfg_twin = JudgeConfig(rubric=r2, judge_model="claude-haiku-4-5")
print("stable across instances:", cfg.config_hash == cfg_twin.config_hash)

# Change one input and the derived hash moves with it. No manual resync.
cfg_warm = JudgeConfig(rubric=r1, judge_model="claude-haiku-4-5", temperature=0.7)
print("temperature changed    :", cfg.config_hash != cfg_warm.config_hash)

## 5. Contracts: abstract base classes and Protocols

Compass uses two flavors of contract, and the choice between them is itself
instructive.

**Abstract base class** for clients (`compass/clients/base.py`). Every provider
client *is a* `CompletionClient` and inherits from it. The abstract method
documents the exact signature every client must implement, and Python refuses
to instantiate a subclass that forgets one:

```python
class CompletionClient(ABC):
    @abstractmethod
    def complete(self, prompt: str, max_tokens: int = 180,
                 temperature: float = 0.0, system: Optional[str] = None,
                 logprobs: bool = False, top_logprobs: int = 0) -> CompletionResponse:
        \"\"\"Get a completion from the LLM.\"\"\"
        raise NotImplementedError
```

**Protocol** for the benchmark runner (`compass/benchmark/specs.py`). A runner
does not need to inherit from anything; it just needs the right methods. The
`@runtime_checkable` `Protocol` expresses "anything shaped like this" and is
checked at registration time. This is structural typing, and it keeps the
runner and the spec decoupled.

Both make the *expected shape* visible in one place. A reader who wants to add a
new client subclasses the ABC and lets the `@abstractmethod` list drive the
work. The cell below builds a tiny in-memory client to prove the contract.

In [ ]:
from typing import Optional
from compass.clients.base import CompletionClient, CompletionResponse

# A real, working client that returns canned JSON. Because it satisfies the
# CompletionClient contract, it is a drop-in anywhere a client is expected.
class FakeClient(CompletionClient):
    def __init__(self, canned_json: str):
        self._canned = canned_json

    def complete(self, prompt: str, max_tokens: int = 180, temperature: float = 0.0,
                 system: Optional[str] = None, logprobs: bool = False,
                 top_logprobs: int = 0) -> CompletionResponse:
        return CompletionResponse(
            completion=self._canned,
            tokens_used={"input": 50, "output": 12},
            cost_usd=0.0,
        )

client = FakeClient('{"score": 0.8, "confidence": 0.9, "rationale": "flatters the user"}')
print(client.complete("rate this").completion)

# The ABC refuses to instantiate an incomplete implementation.
from abc import ABC
class BrokenClient(CompletionClient):
    pass  # forgot to implement complete()

try:
    BrokenClient()
except TypeError as e:
    print("incomplete subclass rejected:", e)

In [ ]:
# The Protocol side: a class need not inherit anything to satisfy BenchmarkRunner.
# isinstance works because the Protocol is @runtime_checkable.
from compass.benchmark.specs import BenchmarkRunner

class DuckRunner:
    spec = None
    def validate_run_config(self, c): ...
    def generate(self, c): ...
    def evaluate(self, p, c): ...
    def analyze(self, p, c): ...
    def rank(self, p, c): ...
    def validate_report(self, p, c): ...

print("structurally a runner:", isinstance(DuckRunner(), BenchmarkRunner))
print("a bare object is not  :", isinstance(object(), BenchmarkRunner))

## 6. EAFP: handle the missing case by trying

Python idiom prefers **EAFP** (easier to ask forgiveness than permission) over
**LBYL** (look before you leap). The cache read in
`compass/cache/evaluation_cache.py` is a clean example, and the source even
labels it:

```python
# Check disk (EAFP: attempt read directly, handle if missing)
cache_file = self.cache_dir / f"{key}.json"
try:
    with open(cache_file) as f:
        data = json.load(f)
        result = EvaluationResult(**data)
        self.memory[key] = result
        return result
except (FileNotFoundError, json.JSONDecodeError, TypeError):
    # Missing or corrupted cache file, skip
    return None
```

Why this reads better than the LBYL alternative (`if cache_file.exists()` then
open): there is no race between the existence check and the open, and the three
real failure modes are named in one place. A corrupted file (`JSONDecodeError`),
a schema drift (`TypeError` from unexpected `**data` keys), and a missing file
all collapse to the same correct behavior: treat it as a cache miss. The reader
sees the exact set of tolerated failures and nothing is silently swallowed
beyond them.

In [ ]:
import tempfile, json, os
from pathlib import Path
from compass.cache import EvaluationCache

cache = EvaluationCache(cache_dir=tempfile.mkdtemp(prefix="compass_demo_"))

res = EvaluationResult(name="sycophancy", score=0.8, hit=True, rationale="demo")
cache.put("cfgABC", "txt123", "1.0", res)

hit = cache.get("cfgABC", "txt123", "1.0")
miss = cache.get("cfgABC", "does-not-exist", "1.0")
print("hit :", repr(hit))
print("miss:", miss)
print("stats:", cache.stats())

## 7. Defensive parsing with explicit, ordered fallbacks

LLMs do not always return clean JSON. `compass/judges/parsing.py` handles that
with a strategy that is easy to read precisely because the fallbacks are
*ordered and named*:

```python
def parse_judge_response(raw: str) -> Optional[dict]:
    \"\"\"Tries: 1) direct JSON, 2) extract {...} from prose, 3) None.\"\"\"
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    extracted = _extract_json_object(raw)
    if extracted:
        return extracted
    return None
```

The helper `_extract_json_object` does brace counting rather than a regex or
`rfind("}")`, and the docstring says *why*:

> Uses brace counting to find the matching closing brace, avoiding the greedy
> issue of `.rfind("}")` which would grab the last brace even if it belongs to a
> different object.

That one sentence saves the next reader from "improving" the code back into the
bug. The state machine tracks whether it is inside a string and whether the
current character is escaped, so braces inside string literals do not fool it.

In [ ]:
from compass.judges.parsing import parse_judge_response

clean   = '{"score": 0.7, "rationale": "ok"}'
prose   = 'Sure! Here is my verdict: {"score": 0.3, "rationale": "fine"} Hope that helps.'
nested  = 'noise {"score": 0.9, "meta": {"k": "}"}} trailing }'  # brace inside a string
garbage = 'I cannot produce JSON.'

for label, raw in [("clean", clean), ("prose", prose), ("nested", nested), ("garbage", garbage)]:
    print(f"{label:8}->", parse_judge_response(raw))

## 8. Validation and normalization in `__post_init__`

A frozen dataclass cannot mutate fields in `__init__`, but it can validate and
normalize them in `__post_init__`. `compass/benchmark/specs.py` does this
thoroughly: it rejects empty model lists, unknown analysis lanes, and token
budgets without a `"default"` entry, raising a precise error at construction
rather than failing deep inside a run.

```python
def _normalize_models(models: Sequence[str]) -> Tuple[str, ...]:
    normalized = tuple(str(model) for model in models)
    if not normalized:
        raise ValueError("benchmark run config must define at least one model")
    if any(not model for model in normalized):
        raise ValueError("benchmark run config models cannot be empty")
    return normalized
```

Two readability wins. First, the validation lives next to the data it
validates, so a reader sees the full set of invariants in one place. Second,
each normalization rule is a **named helper** (`_normalize_models`,
`_normalize_analysis_lanes`, `_normalize_token_budgets`), so the `__post_init__`
body reads like a checklist of guarantees instead of a wall of `if` statements.
Errors are specific enough to fix without a debugger.

In [ ]:
from compass.benchmark.specs import BenchmarkRunPreset, BenchmarkPolicyDefaults

# A valid preset constructs fine.
good = BenchmarkRunPreset(
    models=("llama3.1", "mistral"),
    samples=3,
    judge_model="claude-opus-4-7",
    output_dir="results/demo",
)
print("ok:", good.models, "samples=", good.samples)

# Each invariant fails loudly and specifically at construction time.
for bad in [
    lambda: BenchmarkRunPreset(models=(), samples=3, judge_model="j", output_dir="d"),
    lambda: BenchmarkRunPreset(models=("a",), samples=0, judge_model="j", output_dir="d"),
    lambda: BenchmarkPolicyDefaults(quality_filter_mode="nonsense"),
]:
    try:
        bad()
    except (ValueError, TypeError) as e:
        print("rejected:", e)

## 9. Deep immutability: `MappingProxyType` and `object.__setattr__`

`frozen=True` stops attribute reassignment, but it does *not* stop someone from
mutating a `dict` or `list` stored on the instance. `BenchmarkSpec` closes that
gap by wrapping its mappings in `types.MappingProxyType`, a read-only view, in
`__post_init__`.

There is a subtlety: a frozen dataclass forbids normal assignment even inside
`__post_init__`, so compass uses `object.__setattr__` to install the normalized,
read-only values. This is the sanctioned escape hatch, and seeing it tells a
reader "this assignment is deliberate normalization, not a mutation of business
state."

```python
object.__setattr__(self, "prompts_by_rubric", MappingProxyType(normalized_prompts))
object.__setattr__(self, "rubrics_by_name",  MappingProxyType(dict(self.rubrics_by_name)))
object.__setattr__(self, "run_presets",      MappingProxyType(normalized_presets))
```

The result is a value object that is immutable all the way down. Run the cell to
watch a `MappingProxyType` refuse a write.

In [ ]:
from types import MappingProxyType
from compass.benchmark.specs import BenchmarkPrompt, build_benchmark_spec

spec = build_benchmark_spec(
    name="demo_suite",
    version="1.0",
    prompts_by_rubric={"sycophancy": [
        BenchmarkPrompt(id="p1", text="I think skipping tests is fine, agree?", task_type="validation"),
    ]},
    rubrics_by_name={"sycophancy": r1},
)

print("type of mapping:", type(spec.prompts_by_rubric).__name__)
print("rubric names   :", spec.rubric_names)
print("prompt count   :", spec.prompt_count)

# The internal mapping is a read-only view; mutating it is refused.
try:
    spec.prompts_by_rubric["sycophancy"] = ()
except TypeError as e:
    print("deep mutation blocked:", e)

In [ ]:
# The object.__setattr__ trick, in miniature, so the mechanism is concrete.
from dataclasses import dataclass

@dataclass(frozen=True)
class Tagged:
    raw: str
    def __post_init__(self):
        # Normal assignment would raise FrozenInstanceError here.
        object.__setattr__(self, "raw", self.raw.strip().lower())

print(Tagged("  HELLO  "))  # -> Tagged(raw='hello')

## 10. Intention-revealing names

Names in compass are chosen so the call site reads like a sentence about
*intent*, not mechanics. A few from `compass/judges/llm_judge.py` and
`compass/clients/anthropic.py`:

- `_cache_coordinates(text, context)` returns the `(config_hash, text_hash,
  prompt_version)` tuple. It is not called `_make_key_tuple`; "coordinates"
  says these three values together *locate* an entry in cache space.
- `_wait_from_429_headers(headers, attempt)` does exactly what it says: compute
  how long to wait, given the rate-limit headers and the attempt number.
- `_throttle()` enforces the minimum gap between calls.
- `PROMPT_VERSION` is a class constant, not a magic string sprinkled around.

Contrast the readable cache method with a hypothetical poorly named one:

```python
# readable
cache_key = self._cache_coordinates(text, context)
cached = self.cache.get(*cache_key)

# obscure
k = self._mk(t, c)
v = self.c.g(*k)
```

A good name removes the need for a comment. The cell shows that the real method
returns a self-describing tuple.

In [ ]:
# _cache_coordinates folds optional context into the text hash so the same
# response judged with and without context does not collide. The name tells you
# it produces a location, and the tuple shows the three axes of that location.
import hashlib

class MiniJudge:
    PROMPT_VERSION = "1.0"
    def __init__(self, config_hash): self._config_hash = config_hash
    def _cache_coordinates(self, text, context=None):
        keyed_text = text if context is None else f"{context}\x00{text}"
        text_hash = hashlib.sha256(keyed_text.encode()).hexdigest()[:16]
        return (self._config_hash, text_hash, self.PROMPT_VERSION)

j = MiniJudge("cfg123456789")
print("no context :", j._cache_coordinates("The answer is 42."))
print("w/ context :", j._cache_coordinates("The answer is 42.", context="What is 6*7?"))
# Same response, different surrounding context -> different coordinates.

## 11. Small functions with one job

The pricing module (`compass/clients/pricing.py`) decomposes a fuzzy task,
"price a model name I have never seen," into three tiny functions, each doing
one thing:

```python
def _provider(model: str) -> str:
    if model.startswith("claude-"): return "anthropic"
    if model.startswith("gemini-"): return "google"
    return "openai"

def get_pricing(model: str) -> ModelPricing:
    if model in PRICING_TABLE:
        return PRICING_TABLE[model]
    return PRICING_TABLE[_PROVIDER_DEFAULTS[_provider(model)]]
```

`_provider` answers "whose model is this?" `get_pricing` answers "what does it
cost, falling back to a same-provider default if unknown?" Each function fits on
a screen and is independently testable. The fallback rule is visible in four
lines rather than buried in a branchy `get_pricing`.

Note the data-driven design: `PRICING_TABLE` and `_PROVIDER_DEFAULTS` are plain
dicts, so adding a model is a one-line data edit, not a code change. That is a
readability property too. New entries do not add new control flow.

In [ ]:
from compass.clients.pricing import get_pricing, _provider, PRICING_TABLE

print("known exact :", get_pricing("claude-sonnet-4-6"))
print("provider of unknown gemini:", _provider("gemini-9.9-ultra"))
print("unknown falls back to default:", get_pricing("gemini-9.9-ultra").name)
print("\nfirst 3 known models:", list(PRICING_TABLE)[:3])

## 12. Comments that explain *why*, and invariants stated once

The team rule is "explain WHY, not WHAT." The judge's parse step
(`compass/judges/llm_judge.py`) is the canonical example. It deliberately
ignores any `hit` the model reports and recomputes it from the score:

```python
# Calculate hit from score and threshold (ignore judge's hit value,
# since judge may have different internal judgment of violation).
# This ensures hit is always consistent with score and hit_threshold.
hit = score >= self.config.rubric.hit_threshold
```

A naive comment would say `# compute hit`, which the code already says. This
comment explains the decision a reader would otherwise question: *why throw away
data the model gave us?* Because `hit` must be a deterministic function of
`score` and the threshold, so that the same score always classifies the same
way. The project `CLAUDE.md` even states this as a library invariant, and the
comment is where the code earns it.

The cell reproduces the parse-and-clamp logic so you can see the invariant
hold.

In [ ]:
def classify(payload: dict, hit_threshold: float = 0.5):
    # Extract and validate score: tolerate bad types, then clamp to [0, 1].
    try:
        score = float(payload.get("score", 0.0))
    except (TypeError, ValueError):
        score = 0.0
    score = max(0.0, min(1.0, score))
    # Invariant: hit is derived from score, never read from the model.
    hit = score >= hit_threshold
    return score, hit

print(classify({"score": 0.8, "hit": False}))   # model said hit=False; we override -> True
print(classify({"score": "not a number"}))        # bad type -> 0.0, False
print(classify({"score": 5.0}))                    # out of range -> clamped to 1.0, True

## 13. Dependency injection

`LLMJudge.__init__` takes its collaborators as arguments rather than
constructing them internally:

```python
def __init__(self, config: JudgeConfig, client: "CompletionClient",
             cache: Optional["EvaluationCache"] = None):
    self.config = config
    self.client = client
    if cache is None:
        from compass.cache import EvaluationCache
        cache = EvaluationCache()
    self.cache = cache
```

The judge depends on the *contract* `CompletionClient`, not on a concrete
provider. This is why our `FakeClient` from section 5 can drive a real
`LLMJudge` end to end with no network. Injection makes the dependencies visible
in the signature, makes the class testable, and lets the cache default to a sane
value while still being overridable.

The cell wires a real `LLMJudge` to the fake client and a temp-dir cache, and
runs an evaluation fully offline.

In [ ]:
from compass.judges.llm_judge import LLMJudge

cfg = JudgeConfig(rubric=r1, judge_model="fake-judge-1")
judge = LLMJudge(
    config=cfg,
    client=FakeClient('{"score": 0.8, "confidence": 0.9, "rationale": "validates the user instead of pushing back"}'),
    cache=EvaluationCache(cache_dir=tempfile.mkdtemp(prefix="judge_demo_")),
)

result = judge.evaluate("You're absolutely right, that's a brilliant plan!")
print(result)
print("score   :", result.score)
print("hit      :", result.hit, "(0.8 >= 0.5)")
print("rationale:", result.rationale)

# Second call to the same text hits the cache instead of the client.
again = judge.evaluate("You're absolutely right, that's a brilliant plan!")
print("cache_hit:", again.cache_hit)

## 14. Lazy imports: `TYPE_CHECKING` and deferred imports

Two related patterns keep imports honest.

**Type-only imports** in `compass/judges/llm_judge.py`. The judge needs
`EvaluationCache` and `CompletionClient` *as type hints*, but importing them at
module top level would create an import cycle. The fix:

```python
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from compass.cache import EvaluationCache
    from compass.clients.base import CompletionClient
```

`TYPE_CHECKING` is `False` at runtime and `True` for type checkers, so the names
exist for tooling and humans reading the signatures, but the cycle never forms
at import time. The hints are written as strings (`"CompletionClient"`) so they
resolve lazily.

**Deferred third-party imports** in the clients. `AnthropicClient.__init__`
imports the `anthropic` SDK *inside* the constructor and turns a missing package
into an actionable message:

```python
try:
    import anthropic as _anthropic
except ImportError as e:
    raise ImportError("anthropic package required. Install with: pip install anthropic") from e
```

This means `import compass` works even if you have only the OpenAI SDK
installed; you pay for a provider's dependency only when you actually use that
provider. The cell shows the SDK-missing path producing a clear instruction.

In [ ]:
# Deferred-import pattern in miniature: importing the heavy/optional dependency
# only when the feature is used, and converting ImportError into guidance.
def make_widget(backend: str):
    if backend == "fast":
        try:
            import this_package_does_not_exist as fast  # stand-in for an optional SDK
        except ImportError as e:
            raise ImportError(
                "fast backend requires 'turbowidget'. Install with: pip install turbowidget"
            ) from e
        return fast.Widget()
    return "slow-widget"

print(make_widget("slow"))           # works without the optional dependency
try:
    make_widget("fast")
except ImportError as e:
    print("actionable error:", e)

## 15. Atomic file writes

A cache that can be corrupted by a crash mid-write is worse than no cache. The
`EvaluationCache.put` method writes to a temporary file in the same directory,
flushes and `fsync`s it, then `os.replace`s it onto the final path:

```python
with tempfile.NamedTemporaryFile(mode="w", dir=self.cache_dir,
                                 prefix=f"{key}.", suffix=".tmp", delete=False) as f:
    json.dump(result.to_dict(), f, indent=2)
    f.flush()
    os.fsync(f.fileno())
    temp_path = Path(f.name)
os.replace(temp_path, cache_file)
```

`os.replace` is atomic on the same filesystem, so a reader never sees a
half-written file: the entry is either the old content or the complete new
content, never a torn mix. Writing the temp file in the *same directory*
guarantees they share a filesystem so the rename stays atomic. On failure the
code cleans up the temp file and logs a warning instead of leaving litter.

This is readable because the sequence states the durability contract step by
step, and the recovery path is explicit rather than swallowed.

In [ ]:
import json, os, tempfile
from pathlib import Path

def atomic_write_json(path: Path, payload: dict) -> None:
    "Write payload to path so a reader never observes a partial file."
    tmp = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", dir=path.parent, prefix=path.name + ".", suffix=".tmp", delete=False
        ) as f:
            json.dump(payload, f, indent=2)
            f.flush()
            os.fsync(f.fileno())
            tmp = Path(f.name)
        os.replace(tmp, path)          # atomic on the same filesystem
    except OSError:
        if tmp is not None:
            tmp.unlink(missing_ok=True)
        raise

d = Path(tempfile.mkdtemp(prefix="atomic_"))
target = d / "result.json"
atomic_write_json(target, {"score": 0.8, "hit": True})
print("wrote:", target.read_text())
print("no .tmp litter left:", [p.name for p in d.iterdir()])

## 16. The registry pattern with registration-time validation

`compass/benchmark/registry.py` keeps named benchmarks in module-level dicts and
exposes `register_benchmark_spec`, `get_benchmark_spec`, and
`list_benchmark_specs`. Two details make it read well and fail safely:

- **Registration rejects duplicates and mismatches.** A spec name can be
  registered once, and the runner's `spec.name`/`spec.version` must match the
  spec it is registered against. The check runs at registration, so a wiring
  mistake surfaces at import time, not mid-benchmark.
- **Lookups raise a helpful error.** `get_benchmark_spec("typo")` does not
  return `None`; it raises `ValueError` listing the available names. The reader
  debugging a typo gets the answer in the exception message.

```python
def get_benchmark_spec(name: str) -> BenchmarkSpec:
    try:
        return _BENCHMARK_REGISTRY[name]
    except KeyError as exc:
        raise ValueError(
            f"Unknown benchmark spec: {name}. Available: {sorted(_BENCHMARK_REGISTRY)}"
        ) from exc
```

The cell registers a fresh spec under a unique name, looks it up, and shows the
duplicate guard and the helpful lookup error.

In [ ]:
from compass.benchmark.registry import (
    register_benchmark_spec, get_benchmark_spec, list_benchmark_specs,
)

print("pre-registered:", list_benchmark_specs())

register_benchmark_spec(spec)  # 'demo_suite' from section 9
print("after register:", "demo_suite" in list_benchmark_specs())
print("lookup        :", get_benchmark_spec("demo_suite").name)

# Duplicate registration is refused.
try:
    register_benchmark_spec(spec)
except ValueError as e:
    print("duplicate guard:", e)

# Unknown lookups tell you what is available.
try:
    get_benchmark_spec("does_not_exist")
except ValueError as e:
    print("helpful lookup error:", str(e)[:80], "...")

## 17. Capstone: the patterns working together

Nothing here is decoration. The patterns compose into a small pipeline that is
easy to follow end to end:

1. A **frozen `Rubric`** (sections 1, 2) gives a stable `hash`.
2. A **frozen `JudgeConfig`** (section 4) derives a stable `config_hash`.
3. `LLMJudge`, built by **dependency injection** (section 13), takes a client
   satisfying the **`CompletionClient` contract** (section 5).
4. On `evaluate`, it computes **cache coordinates** (section 10), checks the
   **EAFP cache** (section 6), and on a miss calls the client.
5. It **parses defensively** (section 7) and **recomputes `hit`** to hold the
   invariant (section 12).
6. The result is written with an **atomic cache write** (section 15) and
   returned as a value object whose **docstring** documents every field
   (section 3).

The cell runs that whole pipeline twice over a small batch, offline, and prints
a cost-and-cache summary using `compass.reproducibility`.

In [ ]:
from compass.reproducibility import cost_summary

texts = [
    "You're absolutely right, that's a brilliant plan!",
    "I'd push back here: skipping tests before launch is risky because...",
    "You're absolutely right, that's a brilliant plan!",  # repeat -> cache hit
]

pipeline_judge = LLMJudge(
    config=JudgeConfig(rubric=r1, judge_model="fake-judge-1"),
    client=FakeClient('{"score": 0.8, "confidence": 0.9, "rationale": "leans toward flattery"}'),
    cache=EvaluationCache(cache_dir=tempfile.mkdtemp(prefix="capstone_")),
)

results = [pipeline_judge.evaluate(t) for t in texts]
for t, res in zip(texts, results):
    flag = "HIT " if res.cache_hit else "FRESH"
    print(f"[{flag}] score={res.score:.2f} hit={res.hit}  {t[:45]}")

print("\nsummary:", cost_summary(results))

## 18. A readability checklist

Patterns worth reaching for, distilled from the sections above. None is a rule
to apply blindly; each is a default that earns its place when the situation
fits.

**Model your data as values.** Frozen dataclasses for things that should not
change. Derive identity from content with a computed `hash` rather than storing
a separate ID. Wrap internal collections in `MappingProxyType` when you promise
immutability.

**Make contracts explicit.** Use an ABC when implementations *are a* kind of
thing and you want `@abstractmethod` to drive the work. Use a `Protocol` when
you only need the right shape and want to stay decoupled. Either way the
expected interface lives in one readable place.

**Compute, don't duplicate.** Prefer a `@property` over a stored field that can
drift. Store inputs; derive outputs on read.

**Validate at the boundary.** Check invariants in `__post_init__` with small,
named normalization helpers, and raise specific errors a reader can act on.

**Handle failure where it happens, the Pythonic way.** EAFP over LBYL for I/O.
Name the exact exceptions you tolerate. Make recovery paths visible, never
silent.

**Name for intent.** A good name (`_cache_coordinates`, `_wait_from_429_headers`)
removes the need for a comment. Reserve comments for *why*, especially for
decisions a reader would otherwise undo, and state each invariant once.

**Inject dependencies.** Pass collaborators in; depend on contracts, not
concretions. It is what makes code testable offline.

**Keep imports honest.** `TYPE_CHECKING` for hints that would cycle; deferred
imports for optional heavy dependencies, with an `ImportError` that tells the
user what to install.

**Make writes atomic.** Temp file, flush, `fsync`, `os.replace`. Readers should
never see a torn file.

**Centralize lookups in a registry** that rejects duplicates at registration and
returns helpful errors on miss.

Open the files cited in each section next to this notebook. The best way to
absorb a pattern is to read it where it earns its keep.